# 🏆 AI Predikcija Svjetskog Prvenstva 2026 — FINALNA VERZIJA

## Arhitektura modela
```
┌─────────────────────────────────────────────────────────────┐
│                    IZVORI PODATAKA                          │
│                                                             │
│  results.csv     → Forma + Head-to-Head (49k utakmica)     │
│  train/test.csv  → SP statistike 2002-2026 (48 timova)     │
│  baseline.csv    → ELO bodovi + baseline vjerojatnosti      │
│  players.csv     → Transfermarkt (47k igrača)              │
│  ranking_2026    → FIFA bodovi Siječanj 2026                │
│  Ozljede         → Ručno uneseni penali na ELO             │
└─────────────────┬───────────────────────────────────────────┘
                  │
                  ▼
┌─────────────────────────────────────────────────────────────┐
│                  KOMBINIRANI SCORE                          │
│                                                             │
│  40% ELO (s injury penalima)                               │
│  20% Recentna forma (temporalno vagana, 15 mj)             │
│  20% Tržišna vrijednost squada (Transfermarkt)             │
│  10% Top 3 igrača (zvijezde)                               │
│  10% Head-to-head (zadnjih 8 godina)                       │
│  (+baseline vjerojatnosti ako postoje za taj meč)          │
└─────────────────┬───────────────────────────────────────────┘
                  │
                  ▼
┌─────────────────────────────────────────────────────────────┐
│           MONTE CARLO SIMULACIJA (1000x)                   │
│                                                             │
│  Grupna faza → 8 najboljih trećih → Round of 32            │
│  → R16 → QF → SF → Finale                                 │
└─────────────────────────────────────────────────────────────┘
```

**Upute:** Pokreni svaku ćeliju redom (Shift+Enter ili ▶)

## 📦 Korak 0 — Instalacija paketa

In [ ]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn -q
print('✅ Paketi instalirani!')

## 📁 Korak 1 — Upload datoteka

| Datoteka | Zip |
|---|---|
| `results.csv` | `archive.zip` |
| `shootouts.csv` | `archive.zip` |
| `train.csv` | `archive__5_.zip` |
| `test.csv` | `archive__5_.zip` |
| `fifa_world_rankings_jan_2026.csv` | `archive__6_.zip` |
| `future_match_probabilities_baseline.csv` | `archive__8_.zip` |
| `players.csv` | `archive__9_.zip` |
| `national_teams.csv` | `archive__9_.zip` |

In [ ]:
from google.colab import files
print('📂 Uploadaj sve datoteke odjednom:')
uploaded = files.upload()
print(f'\n✅ Uploadano {len(uploaded)} datoteka:')
for name in uploaded: print(f'   - {name}')

## 🔧 Korak 2 — Učitavanje i priprema podataka

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════
# 1) UČITAVANJE SVIH DATASETA
# ═══════════════════════════════════════════════════════════
results      = pd.read_csv('results.csv')
shootouts    = pd.read_csv('shootouts.csv')
train        = pd.read_csv('train.csv')
test_2026    = pd.read_csv('test.csv')
ranking_2026 = pd.read_csv('fifa_world_rankings_jan_2026.csv')
baseline     = pd.read_csv('future_match_probabilities_baseline.csv')
tm_players   = pd.read_csv('players.csv')
tm_national  = pd.read_csv('national_teams.csv')

# ═══════════════════════════════════════════════════════════
# 2) ČIŠĆENJE REZULTATA
# Izbacujemo NA retke = budući mečevi bez rezultata
# ═══════════════════════════════════════════════════════════
results['date']   = pd.to_datetime(results['date'])
shootouts['date'] = pd.to_datetime(shootouts['date'])
before = len(results)
results = results.dropna(subset=['home_score', 'away_score'])
results['home_score'] = results['home_score'].astype(int)
results['away_score'] = results['away_score'].astype(int)
print(f'✂️  Izbačeno {before - len(results):,} NA redaka (budući mečevi)')

# ═══════════════════════════════════════════════════════════
# 3) ELO BODOVI iz baseline dataseta
# Realnije od FIFA bodova — uzimaju u obzir stvarne performanse
# ═══════════════════════════════════════════════════════════
elo_dict = {}
for _, row in baseline.iterrows():
    if pd.notna(row['home_elo']): elo_dict[row['home_team']] = float(row['home_elo'])
    if pd.notna(row['away_elo']): elo_dict[row['away_team']] = float(row['away_elo'])

# Popuni timove koji nedostaju u baseline — koristimo FIFA bodove iz test.csv
test_elo_map = test_2026.set_index('team')['fifa_points_pre_tournament'].to_dict()
# Skaliramo FIFA bodove na ELO skalu (FIFA max ~1877 → ELO max ~2100)
elo_scale = 2100 / 1877
for team, fifa_pts in test_elo_map.items():
    if team not in elo_dict and pd.notna(fifa_pts):
        elo_dict[team] = round(fifa_pts * elo_scale, 1)

# Ručno dodaj alternativna imena
name_fixes = {
    "Côte d'Ivoire": 'Ivory Coast',
    'Curaçao':        'Curaçao',
}
for old, new in name_fixes.items():
    if old in elo_dict and new not in elo_dict:
        elo_dict[new] = elo_dict[old]

avg_elo = np.mean(list(elo_dict.values()))

# ═══════════════════════════════════════════════════════════
# 4) OZLJEDE KLJUČNIH IGRAČA — ELO PENALI
# Svaka ozljeда ključnog igrača smanjuje ELO tima
# ═══════════════════════════════════════════════════════════
injury_penalties = {
    'Brazil':       80,   # Rodrygo (ACL) + Estevão (hamstring)
    'Japan':        60,   # Endo + Minamino + Suzuki + Machida
    'Netherlands':  40,   # Simons (ACL) + De Ligt (leđa)
    'Mexico':       30,   # Malagón (Ahilova tetiva) — prvog golmana nema
    'Germany':      20,   # Gnabry van
    'Turkey':       20,   # Güler upitan
    'Argentina':    15,   # Foyth (Ahilova tetiva)
    'Croatia':      10,   # Modrić upitan
}
for team, penalty in injury_penalties.items():
    if team in elo_dict:
        elo_dict[team] -= penalty

# ═══════════════════════════════════════════════════════════
# 5) TRANSFERMARKT — TRŽIŠNA VRIJEDNOST SQUADA
# Za svaki tim uzimamo top 23 igrača po tržišnoj vrijednosti
# i računamo: ukupna vrijednost + vrijednost top 3 zvijezde
# ═══════════════════════════════════════════════════════════
citizenship_map = {
    'England':'England', 'Spain':'Spain', 'France':'France',
    'Brazil':'Brazil', 'Argentina':'Argentina', 'Germany':'Germany',
    'Portugal':'Portugal', 'Netherlands':'Netherlands', 'Belgium':'Belgium',
    'Croatia':'Croatia', 'Uruguay':'Uruguay', 'Colombia':'Colombia',
    'Morocco':'Morocco', 'Senegal':'Senegal', 'Japan':'Japan',
    'South Korea':'South Korea', 'United States':'United States',
    'Mexico':'Mexico', 'Canada':'Canada', 'Australia':'Australia',
    'Switzerland':'Switzerland', 'Austria':'Austria', 'Norway':'Norway',
    'Sweden':'Sweden', 'Scotland':'Scotland', 'Ecuador':'Ecuador',
    'Paraguay':'Paraguay', 'Algeria':'Algeria', 'Tunisia':'Tunisia',
    'Egypt':'Egypt', 'Ghana':'Ghana', 'South Africa':'South Africa',
    'Iran':'Iran', 'Saudi Arabia':'Saudi Arabia', 'Iraq':'Iraq',
    'Jordan':'Jordan', 'Uzbekistan':'Uzbekistan', 'New Zealand':'New Zealand',
    'Qatar':'Qatar', 'Panama':'Panama', 'Czech Republic':'Czech Republic',
    'Turkey':'Turkey', 'Bosnia and Herzegovina':'Bosnia-Herzegovina',
    'Ivory Coast':"Côte d'Ivoire", 'DR Congo':'Congo DR',
    'Haiti':'Haiti', 'Curaçao':'Curaçao', 'Cape Verde':'Cape Verde',
}

squad_values, top3_values, squad_ages = {}, {}, {}
FALLBACK_SV, FALLBACK_T3 = 50.0, 15.0  # za timove bez TM podataka

for team, citizenship in citizenship_map.items():
    tp = tm_players[
        (tm_players['country_of_citizenship'] == citizenship) &
        tm_players['market_value_in_eur'].notna()
    ].nlargest(23, 'market_value_in_eur')

    if len(tp) == 0:
        squad_values[team] = FALLBACK_SV
        top3_values[team]  = FALLBACK_T3
        squad_ages[team]   = 27.0
        continue

    squad_values[team] = tp['market_value_in_eur'].sum() / 1e6
    top3_values[team]  = tp.nlargest(3, 'market_value_in_eur')['market_value_in_eur'].sum() / 1e6
    dob = pd.to_datetime(tp['date_of_birth'], errors='coerce')
    squad_ages[team] = (pd.Timestamp('2026-06-15') - dob).dt.days.mean() / 365.25

MAX_SV = max(squad_values.values())  # England ~1665M
MAX_T3 = max(top3_values.values())   # England ~440M

# ═══════════════════════════════════════════════════════════
# 6) BASELINE VJEROJATNOSTI po utakmici
# Za grupne mečeve imamo gotove predikcije — koristimo ih
# kao dodatni izvor informacija
# ═══════════════════════════════════════════════════════════
baseline_probs = {}
for _, row in baseline.iterrows():
    key = (row['home_team'], row['away_team'])
    baseline_probs[key] = (row['p_home_win'], row['p_draw'], row['p_away_win'])

team_data = test_2026.set_index('team')

# ═══════════════════════════════════════════════════════════
# ISPIS SAŽETKA
# ═══════════════════════════════════════════════════════════
print('\n📊 Učitani podaci:')
print(f'   results.csv:    {len(results):,} utakmica  ({results["date"].min().year}–{results["date"].max().year})')
print(f'   train.csv:      {len(train)} SP timova (2002–2022)')
print(f'   test.csv:       {len(test_2026)} timova za SP 2026')
print(f'   baseline:       {len(baseline)} utakmica s ELO predikcijama')
print(f'   TM igrači:      {len(tm_players):,} igrača')
print(f'   ELO timova:     {len(elo_dict)} (uključujući injury penale)')

print('\n🏆 ELO Rangiranje SP 2026 — TOP 15 (s injury penalima):')
top_elo = sorted([(t,e) for t,e in elo_dict.items() if t in citizenship_map], key=lambda x: -x[1])
for i, (team, elo) in enumerate(top_elo[:15], 1):
    inj = injury_penalties.get(team, 0)
    sv  = squad_values.get(team, 0)
    inj_str = f' 🩹-{inj}' if inj else ''
    print(f'   {i:2}. {team:<25} ELO: {elo:.0f}{inj_str:<8}  Squad: {sv:.0f}M€')

## 🤖 Korak 3 — Treniranje ML modela

Koristimo `train.csv` koji ima podatke o svakom timu na svakom SP od 2002. do 2022.
Model uči **koje kombinacije featura** razlikuju prvake od ostalih timova.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import seaborn as sns

FEATURE_COLS = [
    'fifa_rank_pre_tournament',
    'fifa_points_pre_tournament',
    'squad_total_market_value_eur',
    'squad_avg_age',
    'world_cup_titles_before',
    'world_cup_participations_before',
    'groups_passed_before',
    'round16_before',
    'quarterfinals_before',
    'semifinals_before',
    'finals_before',
    'goals_scored_last_4y',
    'goals_received_last_4y',
    'wins_last_4y',
    'losses_last_4y',
    'draws_last_4y',
    'is_host',
]

def add_derived_features(df):
    """Dodaje izvedene featurе koji pojačavaju signal."""
    df = df.copy()
    # Omjer golova (napad vs obrana)
    df['goal_ratio'] = df['goals_scored_last_4y'] / (df['goals_received_last_4y'] + 1)
    # Omjer pobjeda
    total = df['wins_last_4y'] + df['losses_last_4y'] + df['draws_last_4y'] + 1
    df['win_ratio'] = df['wins_last_4y'] / total
    # Iskustvo score (prolasci u dublje faze vrijede više)
    df['experience_score'] = (
        df['groups_passed_before']  * 1 +
        df['round16_before']        * 2 +
        df['quarterfinals_before']  * 3 +
        df['semifinals_before']     * 4 +
        df['finals_before']         * 5 +
        df['world_cup_titles_before'] * 10
    )
    # Vrijednost po igraču
    df['value_per_player'] = df['squad_total_market_value_eur'] / 23
    return df

ALL_FEATURES = FEATURE_COLS + ['goal_ratio', 'win_ratio', 'experience_score', 'value_per_player']

# Priprema podataka
train_clean = train.dropna(subset=FEATURE_COLS + ['winner'])
X_all = add_derived_features(train_clean[FEATURE_COLS])
y_all = train_clean['winner'].astype(int)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)

# Treniranje 3 modela + cross-validacija
print('⏳ Treniram modele s 5-fold cross-validacijom...\n')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'XGBoost': XGBClassifier(
        n_estimators=500, max_depth=4, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=30, eval_metric='logloss',
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=500, max_depth=6, class_weight='balanced',
        random_state=42, n_jobs=-1
    ),
    'Logistic Regression': LogisticRegression(
        max_iter=2000, class_weight='balanced', random_state=42
    ),
}

model_scores = {}
for name, model in models.items():
    use_scaled = name == 'Logistic Regression'
    X_input = X_scaled if use_scaled else X_all
    scores = cross_val_score(model, X_input, y_all, cv=cv, scoring='roc_auc')
    model.fit(X_input, y_all)
    model_scores[name] = scores.mean()
    print(f'   {name:<22} AUC = {scores.mean():.3f} ± {scores.std():.3f}')
    print(f'   {"":<22} (1.0 = savršen, 0.5 = slučajno pogađanje)\n')

best_name  = max(model_scores, key=model_scores.get)
best_model = models[best_name]
print(f'✅ Pobjednik: {best_name} (AUC: {model_scores[best_name]:.3f})')

## 📊 Korak 4 — Što model smatra najvažnijim?

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=ALL_FEATURES)
    importances = importances.sort_values(ascending=True).tail(15)

    plt.figure(figsize=(10, 6))
    colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(importances)))
    bars = importances.plot(kind='barh', color=colors)
    plt.title(f'Važnost featura — {best_name}', fontsize=13, fontweight='bold')
    plt.xlabel('Relativna važnost')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('TOP 5 najvažnijih faktora za pobjedu na SP:')
    for feat, val in importances.sort_values(ascending=False).head(5).items():
        print(f'   {feat}: {val:.3f}')

## 🌍 Korak 5 — SP 2026 Grupe (Službeni žrijeb)

In [ ]:
# Službeni žrijeb SP 2026
wc2026_groups = {
    'A': ['Mexico',        'South Africa',          'South Korea',  'Czech Republic'],
    'B': ['Canada',        'Bosnia and Herzegovina', 'Qatar',        'Switzerland'],
    'C': ['Brazil',        'Morocco',               'Haiti',        'Scotland'],
    'D': ['United States', 'Paraguay',              'Australia',    'Turkey'],
    'E': ['Germany',       'Curaçao',               'Ivory Coast',  'Ecuador'],
    'F': ['Netherlands',   'Japan',                 'Sweden',       'Tunisia'],
    'G': ['Belgium',       'Egypt',                 'Iran',         'New Zealand'],
    'H': ['Spain',         'Cape Verde',            'Saudi Arabia', 'Uruguay'],
    'I': ['France',        'Senegal',               'Iraq',         'Norway'],
    'J': ['Argentina',     'Algeria',               'Austria',      'Jordan'],
    'K': ['Portugal',      'DR Congo',              'Uzbekistan',   'Colombia'],
    'L': ['England',       'Croatia',               'Ghana',        'Panama'],
}

all_wc_teams = [t for grp in wc2026_groups.values() for t in grp]

print(f'✅ {len(all_wc_teams)} timova u {len(wc2026_groups)} grupa\n')
print(f'{"Grupa":<8} {"Timovi"}')
print('─' * 70)
for grp, teams in wc2026_groups.items():
    elos = [elo_dict.get(t, avg_elo) for t in teams]
    svs  = [squad_values.get(t, 0) for t in teams]
    print(f'  {grp}:    {", ".join(teams)}')
    print(f'         ELO:  {" | ".join(f"{e:.0f}" for e in elos)}')
    print(f'         Squad: {" | ".join(f"{s:.0f}M€" for s in svs)}\n')

## ⚙️ Korak 6 — Simulacijske funkcije

Ovdje definiramo **srce modela** — kako predviđamo ishod svake utakmice.

In [ ]:
TODAY = pd.Timestamp('2026-06-15')

# ───────────────────────────────────────────────────────────
# FORMA — temporalno vagana
# Novije utakmice dobivaju veći weight
# ───────────────────────────────────────────────────────────
def get_form(team, results_df, months=15):
    """
    Računa vaganu formu tima iz zadnjih N mjeseci.
    
    Temporalno vaganje:
      - Utakmica od prije 7 dana   → weight ~1.0 (max)
      - Utakmica od prije 8 mj     → weight ~0.47
      - Utakmica od prije 15 mj    → weight  0.15 (min)
    
    Bodovanje po utakmici:
      - Pobjeda  → 1.0
      - Neriješeno → 0.4
      - Poraz    → 0.0
    """
    cutoff  = TODAY - pd.DateOffset(months=months)
    matches = results_df[
        ((results_df['home_team'] == team) | (results_df['away_team'] == team)) &
        (results_df['date'] >= cutoff)
    ]
    if len(matches) == 0:
        return 0.5  # neutralno ako nema podataka

    total_w, weighted_pts = 0.0, 0.0
    for _, row in matches.iterrows():
        days_ago = (TODAY - row['date']).days
        weight   = max(1.0 - days_ago / (months * 30), 0.15)
        gs = row['home_score'] if row['home_team'] == team else row['away_score']
        gc = row['away_score'] if row['home_team'] == team else row['home_score']
        pts = 1.0 if gs > gc else (0.4 if gs == gc else 0.0)
        weighted_pts += pts * weight
        total_w      += weight

    return weighted_pts / total_w if total_w > 0 else 0.5


# ───────────────────────────────────────────────────────────
# HEAD-TO-HEAD
# ───────────────────────────────────────────────────────────
def get_h2h(team1, team2, results_df, years=8):
    """
    Omjer pobjeda tima1 u međusobnim dvobojima zadnjih N godina.
    Vraća 0.5 ako nema dovoljno podataka (min 2 utakmice).
    """
    cutoff = TODAY - pd.DateOffset(years=years)
    h2h = results_df[
        (results_df['date'] >= cutoff) &
        (((results_df['home_team'] == team1) & (results_df['away_team'] == team2)) |
         ((results_df['home_team'] == team2) & (results_df['away_team'] == team1)))
    ]
    if len(h2h) < 2:
        return 0.5
    wins = sum(
        (r['home_team'] == team1 and r['home_score'] > r['away_score']) or
        (r['away_team'] == team1 and r['away_score'] > r['home_score'])
        for _, r in h2h.iterrows()
    )
    return wins / len(h2h)


# ───────────────────────────────────────────────────────────
# GLAVNI PREDIKTOR UTAKMICE
# ───────────────────────────────────────────────────────────
def predict_match(t1, t2, results_df, elo_d, avg_e,
                  bl_probs, sq_vals, t3_vals, max_sv, max_t3):
    """
    Predviđa (p_win1, p_draw, p_win2) za utakmicu t1 vs t2.

    KOMBINIRANI SCORE (težine):
      40% — ELO (realni bodovi + injury penali)
      20% — Forma zadnjih 15 mj (temporalno vagana)
      20% — Tržišna vrijednost squada (Transfermarkt)
      10% — Vrijednost top 3 zvijezde
      10% — Head-to-head (zadnjih 8 god)
      (+10% baseline ako postoji za taj meč, ostalo se skalira)
    """
    # 1) ELO — logistička funkcija
    e1, e2 = elo_d.get(t1, avg_e), elo_d.get(t2, avg_e)
    p_elo1 = 1 / (1 + np.exp(-(e1 - e2) * 5.5 / 400))

    # 2) Forma
    f1, f2 = get_form(t1, results_df), get_form(t2, results_df)
    p_form1 = (f1 + 1e-6) / (f1 + f2 + 2e-6)

    # 3) Tržišna vrijednost squada
    sv1 = sq_vals.get(t1, max_sv * 0.05)
    sv2 = sq_vals.get(t2, max_sv * 0.05)
    p_sv1 = (sv1 + 1) / (sv1 + sv2 + 2)

    # 4) Top 3 zvijezde
    t1v = t3_vals.get(t1, 0)
    t2v = t3_vals.get(t2, 0)
    p_top3 = (t1v + 1) / (t1v + t2v + 2)

    # 5) Head-to-head
    p_h2h = get_h2h(t1, t2, results_df)

    # 6) Baseline vjerojatnost (ako postoji)
    bl1, bl2 = (t1, t2), (t2, t1)
    if bl1 in bl_probs:
        bw1, _, bw2 = bl_probs[bl1]
        p_bl = bw1 / (bw1 + bw2 + 1e-9)
        w_bl = 0.10
    elif bl2 in bl_probs:
        bw2, _, bw1 = bl_probs[bl2]
        p_bl = bw1 / (bw1 + bw2 + 1e-9)
        w_bl = 0.10
    else:
        p_bl, w_bl = 0.5, 0.0

    # Kombinirani weighted score
    r = 1.0 - w_bl  # ostatak za ostale signale
    combined = (
        r * 0.40 * p_elo1  +
        r * 0.20 * p_form1 +
        r * 0.20 * p_sv1   +
        r * 0.10 * p_top3  +
        r * 0.10 * p_h2h   +
        w_bl      * p_bl
    )
    combined = np.clip(combined, 0.04, 0.96)

    # Vjerojatnost neriješenog — veća za ujednačene parove (max ~27%)
    closeness = 1 - abs(combined - 0.5) * 2
    p_draw    = 0.27 * closeness
    return combined * (1 - p_draw), p_draw, (1 - combined) * (1 - p_draw)


# ───────────────────────────────────────────────────────────
# SIMULACIJA GRUPE
# ───────────────────────────────────────────────────────────
def simulate_group(teams, results_df, elo_d, avg_e, bl_probs,
                   sq_vals, t3_vals, max_sv, max_t3):
    """
    Simulira grupnu fazu (svaki sa svakim).
    Vraća: sortirana tablica (pts → gd → gf) i log utakmica.
    """
    st = {t: {'pts':0,'gf':0,'ga':0,'gd':0,'w':0,'d':0,'l':0} for t in teams}
    log = []

    for i in range(len(teams)):
        for j in range(i+1, len(teams)):
            t1, t2 = teams[i], teams[j]
            p1, pd_, p2 = predict_match(t1, t2, results_df, elo_d, avg_e,
                                         bl_probs, sq_vals, t3_vals, max_sv, max_t3)
            out = np.random.choice([0, 1, 2], p=[p1, pd_, p2])

            # Realistična raspodjela golova
            if out == 0:    # t1 pobjeđuje
                g1 = np.random.choice([1,2,3,4], p=[0.28,0.40,0.22,0.10])
                g2 = min(np.random.choice([0,1,2], p=[0.48,0.37,0.15]), g1-1)
                st[t1]['pts']+=3; st[t1]['w']+=1; st[t2]['l']+=1
            elif out == 1:  # neriješeno
                g1 = np.random.choice([0,1,2,3], p=[0.22,0.48,0.22,0.08])
                g2 = g1
                st[t1]['pts']+=1; st[t1]['d']+=1
                st[t2]['pts']+=1; st[t2]['d']+=1
            else:           # t2 pobjeđuje
                g2 = np.random.choice([1,2,3,4], p=[0.28,0.40,0.22,0.10])
                g1 = min(np.random.choice([0,1,2], p=[0.48,0.37,0.15]), g2-1)
                st[t2]['pts']+=3; st[t2]['w']+=1; st[t1]['l']+=1

            st[t1]['gf']+=g1; st[t1]['ga']+=g2; st[t1]['gd']+=g1-g2
            st[t2]['gf']+=g2; st[t2]['ga']+=g1; st[t2]['gd']+=g2-g1
            log.append({'home':t1,'away':t2,'hg':g1,'ag':g2})

    return sorted(st.items(), key=lambda x:(x[1]['pts'],x[1]['gd'],x[1]['gf']), reverse=True), log


# ───────────────────────────────────────────────────────────
# NOKAUT UTAKMICA
# ───────────────────────────────────────────────────────────
def sim_knockout(t1, t2, results_df, elo_d, avg_e, bl_probs,
                  sq_vals, t3_vals, max_sv, max_t3):
    """
    Simulira nokaut utakmicu — mora biti pobjednika.
    Pri neriješenom: penalti gdje ELO favorit ima prednost.
    """
    p1, pd_, p2 = predict_match(t1, t2, results_df, elo_d, avg_e,
                                  bl_probs, sq_vals, t3_vals, max_sv, max_t3)
    out = np.random.choice([0, 1, 2], p=[p1, pd_, p2])
    if out == 0: return t1
    if out == 2: return t2
    # Penalti
    e1, e2 = elo_d.get(t1, avg_e), elo_d.get(t2, avg_e)
    return t1 if np.random.random() < e1 / (e1 + e2) else t2


def run_round(teams, results_df, elo_d, avg_e, bl_probs,
               sq_vals, t3_vals, max_sv, max_t3):
    """Prolazi jednu nokaut rundu — svaki par → pobjednik."""
    return [
        sim_knockout(teams[i], teams[i+1], results_df, elo_d, avg_e,
                      bl_probs, sq_vals, t3_vals, max_sv, max_t3)
        for i in range(0, len(teams)-1, 2)
    ]


print('✅ Simulacijske funkcije definirane!')
print()
print('📊 Provjera snage ključnih timova:')
print(f'{"Tim":<22} {"ELO":>6}  {"Forma":>6}  {"Squad":>8}  {"Top3":>7}')
print('─'*55)
check = ['France','Argentina','Spain','Brazil','England','Germany',
         'Portugal','Netherlands','Japan','Tunisia','Haiti','Jordan']
for t in check:
    e  = elo_dict.get(t, avg_elo)
    f  = get_form(t, results)
    sv = squad_values.get(t, 0)
    t3 = top3_values.get(t, 0)
    inj = f" 🩹" if t in injury_penalties else ""
    print(f'  {t:<20}{inj}  {e:>6.0f}  {f*100:>5.0f}%  {sv:>7.0f}M€  {t3:>6.0f}M€')

## 🎲 Korak 7 — Monte Carlo simulacija

Simuliramo cijeli turnir **1000 puta** i pratimo:
- Koliko puta je svaki tim bio prvak
- Koliko puta je stigao do finala, polufinala, četvrtfinala
- Koliko puta je prošao grupnu fazu

Rezultat: **realne statističke vjerojatnosti** za svaki ishod.

In [ ]:
N_SIM = 1000

# Brojaći za svaki tim i svaku fazu
wins     = {t: 0 for t in all_wc_teams}
finals   = {t: 0 for t in all_wc_teams}
semis    = {t: 0 for t in all_wc_teams}
quarters = {t: 0 for t in all_wc_teams}
groups   = {t: 0 for t in all_wc_teams}

ARGS = (results, elo_dict, avg_elo, baseline_probs,
        squad_values, top3_values, MAX_SV, MAX_T3)

print(f'⏳ Pokrećem {N_SIM} simulacija SP 2026...')

for sim in range(N_SIM):
    if sim % 200 == 0:
        print(f'   Simulacija {sim}/{N_SIM}...')

    # ── GRUPNA FAZA ───────────────────────────────────────────
    qualifiers   = []
    third_placed = []

    for grp, teams in wc2026_groups.items():
        standings, _ = simulate_group(teams, *ARGS)
        qualifiers.append(standings[0][0])  # 1. mjesto
        qualifiers.append(standings[1][0])  # 2. mjesto
        third_placed.append((
            standings[2][0],
            standings[2][1]['pts'],
            standings[2][1]['gd'],
            standings[2][1]['gf']
        ))

    # 8 najboljih trećeplasiranih prolaze dalje
    third_placed.sort(key=lambda x: (x[1], x[2], x[3]), reverse=True)
    qualifiers += [t[0] for t in third_placed[:8]]

    for t in qualifiers: groups[t] += 1

    # ── NOKAUT FAZA ───────────────────────────────────────────
    np.random.shuffle(qualifiers)
    r32 = qualifiers[:32]

    r16  = run_round(r32,  *ARGS);  [quarters.__setitem__(t, quarters[t]+1) for t in r16]
    qf   = run_round(r16,  *ARGS)
    sf   = run_round(qf,   *ARGS);  [semis.__setitem__(t, semis[t]+1) for t in sf]
    fin  = run_round(sf,   *ARGS);  [finals.__setitem__(t, finals[t]+1) for t in fin]
    champ = run_round(fin, *ARGS)[0]
    wins[champ] += 1

print(f'\n✅ {N_SIM} simulacija završeno!')

## 📈 Korak 8 — Rezultati i vizualizacija

In [ ]:
# ── TABLICA REZULTATA ──────────────────────────────────────
probs = pd.DataFrame({
    'Tim':              list(wins.keys()),
    'Prvak (%)':        [100*v/N_SIM for v in wins.values()],
    'Finalista (%)':    [100*v/N_SIM for v in finals.values()],
    'Polufinale (%)':   [100*v/N_SIM for v in semis.values()],
    'Četvrtfinale (%)': [100*v/N_SIM for v in quarters.values()],
    'Prolaz grupe (%)': [100*v/N_SIM for v in groups.values()],
}).sort_values('Prvak (%)', ascending=False).reset_index(drop=True)

print('\n' + '═'*72)
print('🏆  AI PREDIKCIJE SP 2026 — FINALNA VERZIJA')
print(f'    Bazirano na {N_SIM} Monte Carlo simulacija')
print('═'*72)
print(f'{"#":<4} {"Tim":<25} {"Prvak":>7} {"Finale":>7} {"Poluf.":>7} {"QF":>7} {"Prolaz":>7}')
print('─'*72)

medals = {0:'🥇', 1:'🥈', 2:'🥉'}
for i, row in probs.head(20).iterrows():
    prefix = medals.get(i, f'{i+1:2}.')
    print(f'{prefix:<4} {row["Tim"]:<25} '
          f'{row["Prvak (%)"]:>6.1f}% '
          f'{row["Finalista (%)"]:>6.1f}% '
          f'{row["Polufinale (%)"]:>6.1f}% '
          f'{row["Četvrtfinale (%)"]:>6.1f}% '
          f'{row["Prolaz grupe (%)"]:>6.1f}%')

champ = probs.iloc[0]
print('─'*72)
print(f'\n🏆 PREDVIĐENI PRVAK: {champ["Tim"]} ({champ["Prvak (%)":.1f}% od {N_SIM} simulacija)')

In [ ]:
# ── VIZUALIZACIJA ──────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
fig.suptitle('🏆 AI Predikcija SP 2026 — Finalna Verzija', fontsize=17, fontweight='bold')

# Graf 1: TOP 15 — vjerojatnost osvajanja naslova
top15 = probs.head(15).sort_values('Prvak (%)')
pal   = plt.cm.plasma(np.linspace(0.15, 0.85, len(top15)))
bars  = axes[0].barh(top15['Tim'], top15['Prvak (%)'], color=pal, height=0.65, edgecolor='white')
for bar, val in zip(bars, top15['Prvak (%)']):
    axes[0].text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9.5, fontweight='bold')
axes[0].set_xlabel('Vjerojatnost osvajanja SP (%)', fontsize=12)
axes[0].set_title('Tko će postati prvak?', fontsize=13, fontweight='bold')
axes[0].set_xlim(0, top15['Prvak (%)'].max() * 1.3)
axes[0].grid(axis='x', alpha=0.3)

# Graf 2: Stacked — prolaz kroz faze TOP 10
top10 = probs.head(10).sort_values('Prvak (%)')
y = np.arange(len(top10))
h = 0.62
axes[1].barh(y, top10['Prolaz grupe (%)'],  height=h, label='Prolaz iz grupe',  color='#3498db', alpha=0.55)
axes[1].barh(y, top10['Četvrtfinale (%)'],  height=h, label='Četvrtfinale',     color='#27ae60', alpha=0.70)
axes[1].barh(y, top10['Polufinale (%)'],    height=h, label='Polufinale',       color='#f39c12', alpha=0.85)
axes[1].barh(y, top10['Finalista (%)'],     height=h, label='Finale',           color='#e74c3c', alpha=0.90)
axes[1].barh(y, top10['Prvak (%)'],         height=h, label='🏆 Prvak',         color='#8e44ad', alpha=1.00)
axes[1].set_yticks(y)
axes[1].set_yticklabels(top10['Tim'])
axes[1].set_xlabel('Vjerojatnost (%)', fontsize=12)
axes[1].set_title('Napredovanje po fazama — TOP 10', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right', fontsize=9)
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('sp2026_predikcije_final.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graf spremljen!')

## 📋 Korak 9 — Grupna faza (ilustrativna simulacija)

In [ ]:
print('\n📋 GRUPNA FAZA — Simulirani rezultati:\n')

for grp, teams in wc2026_groups.items():
    standings, matches = simulate_group(teams, *ARGS)

    print(f'  ⚽ GRUPA {grp}')
    print(f'  {"":2} {"Tim":<25} {"P":>2} {"N":>2} {"G":>2} {"GZ":>4} {"GP":>4} {"GD":>4} {"Bod":>4}')
    print(f'  {"-"*55}')
    for i, (team, s) in enumerate(standings):
        marker = '✅' if i < 2 else ('🔸' if i == 2 else '❌')
        print(f'  {marker} {team:<25} {s["w"]:>2} {s["d"]:>2} {s["l"]:>2} '
              f'{s["gf"]:>4} {s["ga"]:>4} {s["gd"]:>+4} {s["pts"]:>4}')

    print(f'  Utakmice:', end=' ')
    for m in matches:
        print(f'{m["home"]} {m["hg"]}-{m["ag"]} {m["away"]}', end='  |  ')
    print('\n')

print('  ✅ = direktan prolaz  |  🔸 = može proći kao treći  |  ❌ = ispadanje')

## 🔮 Korak 10 — Predvidi konkretnu utakmicu

In [ ]:
# ════════════════════════════════════════════
# PROMIJENI TIMOVE PO ŽELJI!
tim1 = 'Brazil'
tim2 = 'Argentina'
# ════════════════════════════════════════════

p1, pd_, p2 = predict_match(tim1, tim2, results, elo_dict, avg_elo,
                              baseline_probs, squad_values, top3_values, MAX_SV, MAX_T3)

print(f'\n⚽ {tim1}  vs  {tim2}')
print('═'*50)

# Detalji predikcije
e1, e2 = elo_dict.get(tim1, avg_elo), elo_dict.get(tim2, avg_elo)
f1, f2 = get_form(tim1, results), get_form(tim2, results)
sv1, sv2 = squad_values.get(tim1, 0), squad_values.get(tim2, 0)
t3_1, t3_2 = top3_values.get(tim1, 0), top3_values.get(tim2, 0)
h2h = get_h2h(tim1, tim2, results)

print(f'  {"":<18} {tim1:<18} {tim2}')
print(f'  {"ELO":<18} {e1:<18.0f} {e2:.0f}')
print(f'  {"Forma (15 mj)":<18} {f1*100:<17.0f}% {f2*100:.0f}%')
print(f'  {"Squad (M€)":<18} {sv1:<18.0f} {sv2:.0f}')
print(f'  {"Top 3 (M€)":<18} {t3_1:<18.0f} {t3_2:.0f}')
print(f'  {"H2H (8 god.)":<18} {tim1} pob. {h2h*100:.0f}% međusobnih')
inj1 = injury_penalties.get(tim1, 0)
inj2 = injury_penalties.get(tim2, 0)
if inj1: print(f'  🩹 {tim1} injury penal: -{inj1} ELO')
if inj2: print(f'  🩹 {tim2} injury penal: -{inj2} ELO')
print('─'*50)
print(f'  Pobjeda {tim1:<20} {p1*100:.1f}%')
print(f'  Neriješeno                   {pd_*100:.1f}%')
print(f'  Pobjeda {tim2:<20} {p2*100:.1f}%')

ishod = [f'Pobjeda {tim1}', 'Neriješeno', f'Pobjeda {tim2}'][[p1,pd_,p2].index(max(p1,pd_,p2))]
print(f'\n  🔮 Najvjerojatniji ishod: {ishod}')

# Vizualizacija
fig, ax = plt.subplots(figsize=(8, 4))
labels  = [f'Pobjeda\n{tim1}', 'Neriješeno', f'Pobjeda\n{tim2}']
values  = [p1*100, pd_*100, p2*100]
colors_b = ['#2ecc71', '#95a5a6', '#e74c3c']
bars2   = ax.bar(labels, values, color=colors_b, width=0.5, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars2, values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=13, fontweight='bold')
ax.set_ylabel('Vjerojatnost (%)', fontsize=11)
ax.set_title(f'⚽ {tim1} vs {tim2}', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(values)+12)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 💾 Korak 11 — Spremi i preuzmi rezultate

In [ ]:
probs.to_csv('sp2026_rezultati_final.csv', index=False)
print('✅ Rezultati spremljeni u sp2026_rezultati_final.csv')

# Spremi i detaljni sažetak ozljeda i parametara
with open('sp2026_parametri.txt', 'w') as f:
    f.write('SP 2026 AI MODEL — PARAMETRI I NAPOMENE\n')
    f.write('='*50 + '\n\n')
    f.write(f'Broj simulacija: {N_SIM}\n\n')
    f.write('TEŽINE U MODELU:\n')
    f.write('  40% ELO (realni bodovi + injury penali)\n')
    f.write('  20% Forma (zadnjih 15 mj, temporalno vagana)\n')
    f.write('  20% Tržišna vrijednost squada (Transfermarkt)\n')
    f.write('  10% Top 3 igrača (zvijezde)\n')
    f.write('  10% Head-to-head (zadnjih 8 godina)\n')
    f.write('  (+10% baseline gdje dostupno)\n\n')
    f.write('INJURY PENALI:\n')
    for team, pen in injury_penalties.items():
        f.write(f'  {team}: -{pen} ELO\n')

from google.colab import files
files.download('sp2026_rezultati_final.csv')
files.download('sp2026_predikcije_final.png')
files.download('sp2026_parametri.txt')
print('📥 Datoteke preuzete!')

---
## 📚 Kako funkcionira model — kratki pregled

### 1. Kombinirani score
Za svaki par timova računamo **6 signala** i kombiniramo ih s težinama:
```
ELO (40%)     → osnovna snaga tima, uzima u obzir sve dosadašnje rezultate
Forma (20%)   → zadnjih 15 mj, novije utakmice vrijede više (temporalno vaganje)
Squad val (20%)→ tržišna vrijednost top 23 igrača (Transfermarkt)
Top3 (10%)    → ima li tim zvijezde (Mbappé, Vinicius, Bellingham...)
H2H (10%)     → psihološka prednost u međusobnim dvobojima
Baseline (10%)→ gotove ELO predikcije (ako postoje za taj meč)
```

### 2. Monte Carlo simulacija
```
Simuliramo turnir 1000 puta
Svaka utakmica je stohastična — bolji tim ne pobijedi uvijek
Rezultat: statistička distribucija ishoda za svih 48 timova
```

### 3. Ograničenja modela
- Točnost predikcije u nogometu je tipično 60-70% (literatura)
- Nasumični događaji (ozljede u utakmici, suđenje) se ne mogu modelirati
- Ozljede su unesene ručno — promijeni `injury_penalties` po potrebi
- Neriješeno je najtežje predvidjeti (max ~27% prema literaturi)